# Pandas Class Notes: Data Exploration, Cleaning, Key Validation & Sampling

This notebook covers:
1. **Environment Setup & Data Loading**: Importing packages, reading CSV files with datetime parsing, and extracting date features.
2. **Metadata & Exploration**: Examining table structure, calculating summary ratios, and validating `event_id` uniqueness.
3. **Duplicate Detection & Cleaning**: Identifying duplicate event records and removing affected members across all relational tables.
4. **Key Validation**: Creating a reusable function `check_key` to test primary and candidate keys.
5. **Timeline Inspection**: Sorting and viewing chronological event history for a sample member.
6. **Cohort Sampling**: Drawing a 10% representative member sample and preserving referential integrity across tables.
7. **Renewal Analysis**: Measuring eligible renewal outcomes by expiry cohort.
8. **Activity Segmentation**: Comparing renewal outcomes by member activity and renewal type.

## 1. Environment Setup & Library Imports
Import core libraries required for file path handling, numerical operations, and data manipulation.

In [1]:
# Import standard and third-party libraries
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

## 2. Load Datasets & Feature Engineering
Load `members`, `renewals`, and `events` datasets from CSV files with appropriate datetime parsing, and extract date features (`join_year`, `expiry_year`, `expiry_month`).

In [2]:
# Locate the data folder whether the notebook runs from its own folder or the repository root.
data_dir_candidates = [Path("../notebooks"), Path("notebooks"), Path("python/notebooks")]
DATA_DIR = next((path.resolve() for path in data_dir_candidates if (path / "members.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the data files in ../notebooks or python/notebooks.")
print("Available files in directory:", os.listdir(DATA_DIR))

# Read each source table and parse its date columns at load time.
members = pd.read_csv(DATA_DIR / "members.csv", parse_dates=["join_date"])
renewals = pd.read_csv(DATA_DIR / "renewal_opportunities.csv", parse_dates=["expiry_date", "renewal_deadline"])
events = pd.read_csv(DATA_DIR / "member_events.csv", parse_dates=["event_time"])

# Extract date features used by later cohort summaries.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

print(f"Loaded members:  {members.shape}")
print(f"Loaded renewals: {renewals.shape}")
print(f"Loaded events:   {events.shape}")

Available files in directory: ['DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb', 'members.csv', 'renewal_opportunities.csv', 'member_events.csv']
Loaded members:  (8580, 9)
Loaded renewals: (11580, 9)
Loaded events:   (152057, 10)


## 3. Initial Data Inspection
Preview the first few records of each loaded DataFrame to inspect column names and values.

In [3]:
# Preview the first 5 records of each DataFrame
print("Members Preview:")
display(members.head())

print("Renewals Preview:")
display(renewals.head())

print("Events Preview:")
display(events.head())

Members Preview:


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


Renewals Preview:


,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


Events Preview:


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13


## 4. Metadata Exploration & Ratio Checks
Inspect schema, check unique member distribution, calculate average events per member, and verify primary key uniqueness.

In [4]:
# Inspect data types, non-null counts, and memory usage for events
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 152057 entries, 0 to 152056
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   event_id     152057 non-null  str           
 1   member_id    152057 non-null  str           
 2   renewal_id   152057 non-null  str           
 3   event_time   152057 non-null  datetime64[us]
 4   event_type   152057 non-null  str           
 5   channel      152057 non-null  str           
 6   outcome      152057 non-null  str           
 7   detail_code  143033 non-null  str           
 8   amount       152057 non-null  int64         
 9   event_cost   152057 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(7)
memory usage: 11.6 MB


In [ ]:
# Inspect total rows and unique members in events
print("Total events count:", events.shape[0])
print("Unique member IDs in events:", events["member_id"].nunique())
display(events["member_id"].unique())

In [5]:
# Calculate the average number of events per unique member
avg_events_per_member = round((events.shape[0] / events["member_id"].nunique()), 2)
print("Average events per member:", avg_events_per_member)

Average events per member: 23.83


### Verify Primary Key Uniqueness (`event_id`)
Check whether `event_id` is globally unique across all event records (a ratio of 1.0 indicates a strictly unique primary key).

In [ ]:
# Verify uniqueness of event_id
event_id_uniqueness_ratio = events["event_id"].nunique() / events.shape[0]
print("event_id uniqueness ratio (1.0 = strictly unique):", event_id_uniqueness_ratio)

## 5. Duplicate Detection & Data Cleaning
Identify duplicate events matching on `['member_id', 'renewal_id', 'event_time']` and remove all records associated with those affected members across all three tables.

In [ ]:
# Define composite key columns to check for duplicate event occurrences
cols = ["member_id", "renewal_id", "event_time"]

# Calculate the total number of duplicate groupings
duplicate_groups_count = (events.groupby(cols).size() > 1).sum()
print("Total duplicate event groups found:", duplicate_groups_count)

In [ ]:
# Group events by composite key and filter those appearing more than once
# Use .reset_index() to convert the Series into a DataFrame with accessible columns
group = events.groupby(cols).size()
group = group[group > 1].reset_index(name="event_count")

print("Duplicate event groupings:")
display(group)

In [ ]:
# Extract the list of unique member IDs associated with duplicate events
duplicate_member_ids = group["member_id"].unique().tolist()
print(f"Total member IDs with duplicate events to remove: {len(duplicate_member_ids)}")

# Remove duplicate member IDs entirely across events, members, and renewals tables
events = events[~events["member_id"].isin(duplicate_member_ids)]
members = members[~members["member_id"].isin(duplicate_member_ids)]
renewals = renewals[~renewals["member_id"].isin(duplicate_member_ids)]

# Verify cleaned row counts and remaining unique members
print("Post-cleaning dataset summary:")
print(f"Events:   {events.shape[0]} rows (Unique members: {events['member_id'].nunique()})")
print(f"Members:  {members.shape[0]} rows (Unique members: {members['member_id'].nunique()})")
print(f"Renewals: {renewals.shape[0]} rows (Unique members: {renewals['member_id'].nunique()})")

## 6. Primary & Candidate Key Validation
Define a reusable helper function `check_key` to confirm whether a given column or set of columns uniquely identifies rows in a DataFrame.

In [ ]:
def check_key(df: pd.DataFrame, cols: list) -> bool:
    """
    Check whether specified column(s) uniquely identify each row in df.
    Returns True if no duplicates exist, False otherwise.
    """
    group_counts = df.groupby(cols).size()
    return (group_counts > 1).sum() == 0

# Validate primary key for members table
is_member_key_unique = check_key(members, ["member_id"])
print("Is ['member_id'] a unique primary key in members?:", is_member_key_unique)

# Validate composite key for renewals table
is_renewal_key_unique = check_key(renewals, ["member_id", "renewal_number"])
print("Is ['member_id', 'renewal_number'] a unique key in renewals?:", is_renewal_key_unique)

## 7. Member Event Timeline Inspection
Inspect the chronological event history for a specific member (`MP000456`) to observe sequential behavior over time.

In [ ]:
# Filter events for a sample member and sort chronologically by event_time
sample_member_id = "MP000456"
member_timeline = events[events["member_id"] == sample_member_id].sort_values(by="event_time")

print(f"Event timeline for member: {sample_member_id} ({len(member_timeline)} total events)")
display(member_timeline.head(10))

## 8. Cohort Sampling (10% Random Sample)
Draw a 10% random sample of members, and subset `events` and `renewals` using the sampled member IDs to preserve relational consistency.

In [ ]:
# Sample 10% of member IDs from cleaned members table (random_state ensures reproducibility)
sample_member_ids = members["member_id"].sample(frac=0.1, random_state=42).tolist()
print(f"Sampled {len(sample_member_ids)} member IDs out of {len(members)} total.")

# Filter all three tables using the sampled member IDs
sample_members = members[members["member_id"].isin(sample_member_ids)]
sample_renewals = renewals[renewals["member_id"].isin(sample_member_ids)]
sample_events = events[events["member_id"].isin(sample_member_ids)]

print(f"Sampled Members shape:  {sample_members.shape}")
print(f"Sampled Renewals shape: {sample_renewals.shape}")
print(f"Sampled Events shape:   {sample_events.shape}")

In [ ]:
# Preview the sampled datasets
print("Sampled Members Preview:")
display(sample_members.head())

print("Sampled Renewals Preview:")
display(sample_renewals.head())

print("Sampled Events Preview:")
display(sample_events.head())

## 9. Basic Event-Level Renewal Exploration

These quick summaries count event rows. The eligibility-based analysis below is the more appropriate view for renewal performance.


In [ ]:
events["event_time"] = pd.to_datetime(events["event_time"])

In [ ]:
events["event_year"] = events["event_time"].dt.year
events.groupby('event_year').size()

In [ ]:
# Find Renewals
cond = (events["event_type"]=="PAYMENT_ATTEMPT")&(events["outcome"]=="SUCCESS")
events[cond].groupby('event_year').agg(
    members_renewed = ("member_id", "nunique"),
    total_renewals = ("renewal_id", "nunique"),

)

In [ ]:
events.groupby('event_year').agg(members_total = ("member_id", "nunique"),)

In [ ]:
cond= (events['event_type']=='PAYMENT_ATTEMPT')&(events['outcome']=='SUCCESS')

events['is_renewed'] = np.where(cond,1,0)

In [ ]:
events.groupby('event_year').agg(members_total = ("member_id", "nunique"),
                                 total_renewed = ("is_renewed", "sum"),)

*“More members are renewing, but I want to improve the chance that a member renews. I also don’t want us to achieve that simply by giving away membership fees.”*

1. Establish the metric “Our primary metric is the renewal rate.”
- Renewal rate = successful renewals ÷ eligible renewal opportunities

## 10. Eligible Renewal Analysis

The next summaries include only renewal opportunities whose deadline has passed by the reporting cutoff. They group payment activity by event month.


| Rule | Why we need it |
|---|---|
| Compare September expiries across years | Keep the reporting periods comparable |
| Include only completed renewal windows | Give every opportunity its full chance to renew |
| Count success within the valid payment window | Apply one consistent renewal definition |
| Count each opportunity once | Retries must not inflate renewal counts |

In [ ]:
# Inspect renewal deadlines before applying the reporting cutoff.
renewals.info()

In [ ]:
# An opportunity is eligible once its renewal deadline has passed by the reporting cutoff.
REPORTING_CUTOFF = pd.Timestamp('2026-09-09')
renewals['is_eligible'] = renewals['renewal_deadline'].le(REPORTING_CUTOFF)
renewals

In [ ]:
# Attach renewal opportunity fields so event outcomes can be analyzed by expiry cohort.
events_renewal = events.merge(
    renewals[['renewal_id', 'is_eligible', 'expiry_date', 'renewal_fee', 'renewal_number', 'offered_discount']],
    on='renewal_id',
    how='left',
    validate='many_to_one'
)
events_renewal.head()

In [ ]:
# Keep successful eligible payment outcomes, then reduce them to one row per renewal opportunity.
cond_eligible_success = (
    (events_renewal['event_type'] == 'PAYMENT_ATTEMPT') &
    (events_renewal['outcome'] == 'SUCCESS') &
    (events_renewal['is_eligible'])
)
successful_renewals = (
    events_renewal.loc[cond_eligible_success]
    .groupby('renewal_id', as_index=False)
    .agg(
        is_renewed=('is_renewed', 'max'),
        cash_collected=('amount', 'sum'),
        fee_collected=('renewal_fee', 'first'),
        total_discount=('offered_discount', 'first')
    )
)

# Keep every eligible opportunity in the denominator, including those without a successful payment.
eligible_renewals = renewals.loc[renewals['is_eligible']].copy()
eligible_renewals['expiry_year'] = eligible_renewals['expiry_date'].dt.year
eligible_renewals['expiry_month'] = eligible_renewals['expiry_date'].dt.month
eligible_renewals['expiry_year_month'] = eligible_renewals['expiry_date'].dt.strftime('%Y-%m')
eligible_renewals = eligible_renewals.merge(
    successful_renewals, on='renewal_id', how='left', validate='one_to_one'
)

# No matching successful payment means zero renewal and zero realized financials.
metric_columns = ['is_renewed', 'cash_collected', 'fee_collected', 'total_discount']
eligible_renewals[metric_columns] = eligible_renewals[metric_columns].fillna(0)

annual_renewal_summary = eligible_renewals.groupby(['expiry_year', 'expiry_month']).agg(
    eligible_opportunities=('renewal_id', 'nunique'),
    total_renewed=('is_renewed', 'sum'),
    cash_collected=('cash_collected', 'sum'),
    fee_collected=('fee_collected', 'sum'),
    total_discount=('total_discount', 'sum')
)
annual_renewal_summary['renewal_rate_pct'] = (
    annual_renewal_summary['total_renewed'] / annual_renewal_summary['eligible_opportunities'] * 100
).round(2)

print('Eligible Renewal Performance by Expiry Year and Month:')
display(annual_renewal_summary)

## Monthly Renewal & Financial Analysis

Summarize eligible renewal opportunities by expiry month. Each eligible opportunity appears once in the denominator.


### Monthly Renewal Cohorts
Compare successful renewals with eligible opportunities for each expiry month.


In [ ]:
# Summarize each expiry cohort with a consistent opportunity denominator.
monthly_renewal_summary = eligible_renewals.groupby('expiry_year_month').agg(
    eligible_opportunities=('renewal_id', 'nunique'),
    total_renewed=('is_renewed', 'sum'),
    cash_collected=('cash_collected', 'sum'),
    fee_collected=('fee_collected', 'sum'),
    total_discount=('total_discount', 'sum')
)
monthly_renewal_summary['renewal_rate_pct'] = (
    monthly_renewal_summary['total_renewed']
    / monthly_renewal_summary['eligible_opportunities'] * 100
).round(2)

print('Monthly Renewal Summary by Expiry Cohort:')
display(monthly_renewal_summary)

### Summary of Findings
- **September expiry cohorts**: Renewal rates now compare successful renewals with eligible renewal opportunities, grouped by expiry month.
- **Financial realization**: `cash_collected` reflects net payments received (`amount`), while `total_discount` captures promotional reductions.
- **Cutoff**: The 2026 September cohort may be partial because only deadlines on or before the reporting cutoff are eligible.


## 11. Renewal and Activity Segmentation

Create renewal-level and member-activity segments, then compare their renewal outcomes.


In [ ]:
# events_renewal
cond = (events_renewal['event_type'] == 'PAYMENT_ATTEMPT') & (events_renewal['outcome'] == 'SUCCESS') & (events_renewal['is_eligible'] == 1)
events_renewal['renewal_fee_success'] = np.where(cond, events_renewal['renewal_fee'], 0)
events_renewal['total_discount_success'] = np.where(cond, events_renewal['offered_discount'], 0)

In [ ]:
events_renewal.groupby('event_year').agg(total_members = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), fee_collected = ('renewal_fee_success', 'sum'), total_discount = ('total_discount_success', 'sum'))

In [ ]:
events_renewal['type_customer_renewal'] = np.where(events_renewal['renewal_number']>1, 'Later Renewal', 'First Renewal')

In [ ]:
events_renewal.groupby(['event_year', 'type_customer_renewal']).agg(total_members = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), fee_collected = ('renewal_fee_success', 'sum'), total_discount = ('total_discount_success', 'sum'))

In [ ]:
events_renewal.groupby(['event_year', 'renewal_id']).agg(total_members = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), fee_collected = ('renewal_fee_success', 'sum'), total_discount = ('total_discount_success', 'sum'))

In [ ]:
# Use the three months ending 2026-10-10 as the member-activity observation window.
cut_off = pd.to_datetime('2026-10-10') - pd.DateOffset(months=3)
cut_off

In [ ]:
cond = (events_renewal['event_time']>= cut_off) & (events_renewal['event_time']<= pd.to_datetime('2026-10-10')) & (events_renewal['event_type']!= 'PAYMENT_ATTEMPT')
active_members = events_renewal[cond].groupby('member_id').agg(number_of_events = ('event_id', 'count')).reset_index()
active_members

In [ ]:
active_members['type_customer_activity'] = pd.qcut(active_members['number_of_events'], q = 3, labels = ['Low Activity', 'Moderate Activity', 'High Activity'])
active_members['type_customer_activity'] = active_members['type_customer_activity'].astype('object')

In [ ]:
active_members

In [ ]:
# Inspect how event counts are distributed without requiring plotting libraries.
active_members['number_of_events'].value_counts().sort_index()

In [ ]:
all_members = pd.DataFrame(events_renewal['member_id'].unique(), columns = ['member_id'])
all_members

In [ ]:
# Start from one row per member; active_members has one row per active member.
# A fresh left table prevents pandas from adding _x/_y suffixes to stale columns.
all_members = all_members[['member_id']].merge(
    active_members[['member_id', 'number_of_events', 'type_customer_activity']],
    how='left',
    on='member_id',
    validate='one_to_one'
)
all_members

In [ ]:
# Members with no qualifying events are assigned zero events and a separate category.
all_members['number_of_events'] = all_members['number_of_events'].fillna(0)
all_members['type_customer_activity'] = all_members['type_customer_activity'].fillna('No Activity')

In [ ]:
# Add each member's activity category to their event rows.
events_renewals_customer_type = events_renewal.merge(
    all_members, on='member_id', how='left', validate='many_to_one'
)

In [ ]:
events_renewals_customer_type

In [ ]:
# Keep the full merged event table intact and analyze the 2026 slice separately.
events_renewals_2026 = events_renewals_customer_type[
    events_renewals_customer_type['event_year'] == 2026
]
group = events_renewals_2026.groupby('type_customer_activity').agg(
    total_members=('member_id', 'nunique'),
    total_renewed=('is_renewed', 'sum'),
    fee_collected=('renewal_fee_success', 'sum'),
    total_discount=('total_discount_success', 'sum')
)

In [ ]:
group['renewal_rate'] = group['total_renewed']/group['total_members']

group

In [ ]:
group = events_renewals_2026.groupby(
    ['type_customer_activity', 'type_customer_renewal']
).agg(
    total_members=('member_id', 'nunique'),
    total_renewed=('is_renewed', 'sum'),
    fee_collected=('renewal_fee_success', 'sum'),
    total_discount=('total_discount_success', 'sum')
)
group['renewal_rate'] = group['total_renewed'] / group['total_members']
group

In [ ]:
group = events_renewal.groupby(['channel']).agg(total_members = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), fee_collected = ('renewal_fee_success', 'sum'), total_discount = ('total_discount_success', 'sum'))
group['%renewed'] = group['total_renewed']/group['total_members']
group

In [ ]:
group =  events_renewal.groupby(['channel', 'type_customer_renewal']).agg(total_members = ('member_id', 'nunique'), total_renewed = ('is_renewed', 'sum'), fee_collected = ('renewal_fee_success', 'sum'), total_discount = ('total_discount_success', 'sum'))
group['%renewed'] = group['total_renewed']/group['total_members']
group

In [6]:
from scipy import stats

stats.bernoulli(p=0.72)          # single trial
stats.binom(n=600, p=0.72)       # count of successes
stats.norm(loc=18, scale=6)      # normal
stats.norm()                     # standard normal (loc=0, scale=1)